# Wan 2.2 (5B) di Kaggle: video AI gratis tanpa watermark

Notebook ini membuat klip video pendek dari **teks** atau dari **foto produk** dengan model open-source
[Wan2.2-TI2V-5B](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers) (lisensi Apache 2.0, boleh dipakai komersial).
Hasilnya MP4 9:16 tanpa watermark, siap diimpor ke KanvasCut.

## Sebelum mulai (panel kanan → *Settings*)
1. **Accelerator: GPU T4 x2** (paling cepat). GPU P100 atau T4 tunggal juga bisa, tapi lebih lambat.
2. **Internet: On**. Ini butuh akun Kaggle yang sudah verifikasi nomor HP.
3. Jalankan sel dari atas ke bawah. Unduh model sekitar 33 GB dan memuat model butuh sekitar 10–20 menit, hanya sekali per sesi.

## Perkiraan waktu di T4
Belum diukur langsung. Mulai dengan preset `"tes"`, lalu lihat angka detik per step di progress bar.

| Preset | Ukuran | Durasi | Perkiraan |
|---|---|---|---|
| `tes` | 480×832 | ~1 dtk | beberapa menit |
| `cepat` | 480×832 | ~2 dtk | ~5–10 menit |
| `standar` | 480×832 | ~3,4 dtk | ~10–20 menit |
| `hd` | 704×1280 | ~5 dtk | bisa > 45 menit, berisiko kehabisan memori |

Kuota GPU Kaggle terbatas per minggu (cek di profil Kaggle Anda), jadi pilih preset dengan hemat.

**Ingat:** konten berisi klip AI wajib diberi label *AI-generated* di TikTok, dan produk harus tampil akurat.

## 1. Pasang library

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

## 2. Unduh dan muat model
`/kaggle/working` hanya 20 GB, jadi model disimpan di `/tmp`.

In [ ]:
import os
os.environ["HF_HOME"] = "/tmp/hf"   # harus sebelum import huggingface_hub/diffusers

import gc, torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, UMT5EncoderModel
from diffusers import AutoencoderKLWan, WanTransformer3DModel, UniPCMultistepScheduler

assert torch.cuda.is_available(), "GPU belum aktif: Settings → Accelerator → GPU T4 x2"
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    print(f"cuda:{i}", torch.cuda.get_device_name(i))

GEN_DEVICE = "cuda:0"                            # transformer + VAE
TE_DEVICE = "cuda:1" if n_gpu >= 2 else "cpu"    # text encoder UMT5 (bf16)
print("Generator di", GEN_DEVICE, "| text encoder di", TE_DEVICE)

MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
model_dir = snapshot_download(MODEL_ID, ignore_patterns=["assets/*", "examples/*"])

tokenizer = AutoTokenizer.from_pretrained(model_dir, subfolder="tokenizer")
text_encoder = UMT5EncoderModel.from_pretrained(
    model_dir, subfolder="text_encoder", torch_dtype=torch.bfloat16).to(TE_DEVICE).eval()
gc.collect()

# T4/P100 tidak punya bf16 native, jadi transformer pakai fp16.
transformer = WanTransformer3DModel.from_pretrained(
    model_dir, subfolder="transformer", torch_dtype=torch.float16).to(GEN_DEVICE)
gc.collect()

vae = AutoencoderKLWan.from_pretrained(model_dir, subfolder="vae", torch_dtype=torch.float32).to(GEN_DEVICE)
vae.enable_tiling()   # decode per potongan supaya memori GPU tidak melonjak di akhir

scheduler = UniPCMultistepScheduler.from_pretrained(model_dir, subfolder="scheduler")
print(f"Memori GPU terpakai: {torch.cuda.memory_allocated(0)/2**30:.1f} GB di cuda:0")

## 3. Fungsi pembuat video

In [ ]:
import gc, json, random, time
from pathlib import Path

import torch
from PIL import Image, ImageOps
from diffusers import WanPipeline, WanImageToVideoPipeline
from diffusers.utils import export_to_video

NEGATIVE_PROMPT = (
    "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，"
    "JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，"
    "手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
)

# Semua ukuran harus kelipatan 32, jumlah frame = 4k+1. Model dilatih 24 fps.
PRESETS = {
    "tes":     dict(height=832,  width=480, frames=25,  steps=20, cfg=5.0),  # ~1 detik, cek mesin jalan
    "cepat":   dict(height=832,  width=480, frames=49,  steps=25, cfg=5.0),  # ~2 detik
    "standar": dict(height=832,  width=480, frames=81,  steps=30, cfg=5.0),  # ~3,4 detik
    "hd":      dict(height=1280, width=704, frames=121, steps=40, cfg=5.0),  # ~5 detik 720p, sangat lama di T4
}
FPS = 24


def build_pipes(tokenizer, text_encoder, transformer, vae, scheduler):
    """Pipeline encoder terpisah dari pipeline generator.

    Text encoder (UMT5, bf16) tinggal di GPU kedua / CPU, transformer + VAE di GPU pertama.
    Pipeline generator dibuat tanpa text_encoder supaya diffusers memilih cuda:0 sebagai device kerja.
    """
    enc = WanPipeline(tokenizer=tokenizer, text_encoder=text_encoder, vae=None,
                      scheduler=scheduler, transformer=None)
    common = dict(tokenizer=tokenizer, text_encoder=None, vae=vae, scheduler=scheduler,
                  transformer=transformer, expand_timesteps=True)
    return enc, WanPipeline(**common), WanImageToVideoPipeline(**common)


class WanStudio:
    def __init__(self, enc, t2v, i2v, gen_device, te_device, out_dir):
        self.enc, self.t2v, self.i2v = enc, t2v, i2v
        self.gen_device, self.te_device = torch.device(gen_device), torch.device(te_device)
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self._cache = {}

    @torch.no_grad()
    def _encode_on(self, text, device):
        emb, _ = self.enc.encode_prompt(prompt=text, do_classifier_free_guidance=False,
                                        max_sequence_length=512, device=device)
        return emb

    def encode(self, text):
        if text not in self._cache:
            try:
                emb = self._encode_on(text, self.te_device)
                ok = bool(torch.isfinite(emb).all())
            except RuntimeError as e:  # mis. bf16 tidak didukung di GPU ini
                print("Text encoder gagal di", self.te_device, "->", e)
                ok = False
            if not ok and self.te_device.type != "cpu":
                print("Pindah text encoder ke CPU (lebih lambat, tapi aman).")
                self.enc.text_encoder.to("cpu")
                self.te_device = torch.device("cpu")
                gc.collect(); torch.cuda.empty_cache()
                emb = self._encode_on(text, self.te_device)
            self._cache[text] = emb.cpu()
        return self._cache[text].to(self.gen_device)

    def generate(self, prompt, image=None, preset="cepat", seed=None, name=None,
                 negative_prompt=NEGATIVE_PROMPT):
        p = PRESETS[preset]
        seed = random.randrange(2**31) if seed is None else int(seed)
        g = torch.Generator(device=self.gen_device).manual_seed(seed)
        kw = dict(prompt_embeds=self.encode(prompt), negative_prompt_embeds=self.encode(negative_prompt),
                  height=p["height"], width=p["width"], num_frames=p["frames"],
                  num_inference_steps=p["steps"], guidance_scale=p["cfg"], generator=g)
        t0 = time.time()
        if image is None:
            frames = self.t2v(**kw).frames[0]
        else:
            img = image if isinstance(image, Image.Image) else Image.open(image)
            # potong tengah ke rasio video supaya foto tidak gepeng
            img = ImageOps.fit(ImageOps.exif_transpose(img).convert("RGB"),
                               (p["width"], p["height"]), Image.LANCZOS)
            frames = self.i2v(image=img, **kw).frames[0]
        nomor = len(list(self.out_dir.glob("*.mp4"))) + 1
        name = name or f"wan_{time.strftime('%Y%m%d')}_{nomor:03d}_{seed}"
        path = self.out_dir / f"{name}.mp4"
        export_to_video(frames, str(path), fps=FPS)
        info = dict(file=path.name, prompt=prompt, image=str(image) if image else None,
                    preset=preset, seed=seed, seconds=round(time.time() - t0, 1))
        with open(self.out_dir / "log.jsonl", "a") as f:
            f.write(json.dumps(info, ensure_ascii=False) + "\n")
        print(f"✅ {path.name}  ({info['seconds']} dtk, seed {seed})")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return path

    def batch(self, jobs, preset="cepat"):
        hasil = []
        for i, job in enumerate(jobs, 1):
            job = {"prompt": job} if isinstance(job, str) else dict(job)
            print(f"[{i}/{len(jobs)}] {job['prompt'][:80]}")
            try:
                hasil.append(self.generate(preset=job.pop("preset", preset), **job))
            except torch.cuda.OutOfMemoryError:
                print("❌ GPU kehabisan memori. Pakai preset lebih kecil.")
                gc.collect(); torch.cuda.empty_cache()
        return hasil

In [ ]:
enc, t2v, i2v = build_pipes(tokenizer, text_encoder, transformer, vae, scheduler)
studio = WanStudio(enc, t2v, i2v, GEN_DEVICE, TE_DEVICE, out_dir="/kaggle/working/wan_output")
print("Siap. Device kerja:", t2v._execution_device)

## 4. Tes pertama (teks → video)
Prompt bahasa Inggris biasanya memberi hasil paling baik. Jelaskan **subjek, gerakan, kamera, cahaya**.

In [ ]:
from IPython.display import Video

hasil = studio.generate(
    "A white ceramic coffee mug on a wooden table, steam rising, morning sunlight through a window, "
    "slow camera push-in, shallow depth of field, cinematic product shot",
    preset="tes", seed=42,
)
Video(str(hasil), embed=True, width=270)

## 5. Foto produk → video (image-to-video)
1. Upload foto produk sebagai Dataset Kaggle (*Add Input* → *Upload*). Foto akan ada di `/kaggle/input/<nama-dataset>/`.
2. Ganti `FOTO` di bawah dengan path foto Anda.

Foto dipotong tengah ke rasio 9:16. Prompt sebaiknya menggambarkan **gerakan kamera dan suasana**, bukan mengubah produknya.
Periksa hasilnya: logo atau tulisan di produk bisa berubah bentuk, dan video seperti itu jangan dipakai.

In [ ]:
import glob
print("\n".join(glob.glob("/kaggle/input/**/*.*", recursive=True)[:20]) or "Belum ada file di /kaggle/input")

FOTO = None   # contoh: "/kaggle/input/produk-saya/sepatu.jpg"
if FOTO:
    hasil = studio.generate(
        "The product stays still and sharp in the center, slow orbiting camera, soft studio lighting, "
        "subtle light reflections, clean background, commercial product video",
        image=FOTO, preset="cepat",
    )
    display(Video(str(hasil), embed=True, width=270))

## 6. Produksi harian (batch)
Isi daftar di bawah, lalu jalankan. Supaya bisa ditinggal (browser boleh ditutup), pakai
**Save Version → Save & Run All (Commit)**. Hasilnya muncul di tab *Output* versi notebook tersebut.

In [ ]:
PRESET_HARIAN = "cepat"

JOBS = [
    # teks saja
    "Close-up of hands opening a pastel gift box, confetti falling, bright soft light, slow motion, joyful mood",
    "A cozy minimalist bedroom at golden hour, sheer curtains moving in the breeze, slow pan right, warm tones",
    "Fresh orange slices splashing into sparkling water, droplets in slow motion, bright summer light, macro shot",
    # foto produk (isi path-nya dulu), boleh campur dengan preset lain:
    # {"prompt": "slow orbit around the product, soft studio light", "image": "/kaggle/input/produk-saya/tas.jpg"},
    # {"prompt": "...", "preset": "standar", "seed": 7},
]

hasil_batch = studio.batch(JOBS, preset=PRESET_HARIAN)

## 7. Unduh semua hasil sekaligus

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/wan_output", "zip", "/kaggle/working/wan_output")
print("Unduh dari panel Output:", zip_path)
print(open("/kaggle/working/wan_output/log.jsonl").read())